# Phase 3B - Abstention Evaluation (Colab)

Chạy `smoke`, `development`, hoặc `final` trên dataset 200 case đã duyệt. Final bị khóa cho đến khi có `winner_decision.json` từ development.

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='SET_TO_COMMIT_CONTAINING_PHASE3_RUNNER'
HF_ARTIFACT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
HF_ARTIFACT_REVISION='locked-bge-m3-512-64-deduplicated-v2'
HF_ARTIFACT_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
HF_ARTIFACT_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'
HF_PREPARATION_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
HF_PREPARATION_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
HF_PREPARATION_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
HF_PREPARATION_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'
ABSTENTION_DATASET_PATH=''  # Finalized directory or ZIP, uploaded to Colab/Drive.
PHASE2_WINNER_PROMPT_ID='p2'  # Replace only after Phase 2B winner is locked.
CONTEXT_DEPTH=5  # Replace only after Phase 2B winner is locked.
RUN_STAGE='smoke'  # smoke | development | final
EXECUTE=False
WINNER_DECISION_PATH=''  # Required for final.
THRESHOLD_DECISION_PATH=''  # Required for final if not embedded in winner decision.

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,yaml
from google.colab import drive,userdata
drive.mount('/content/drive')
ROOT=Path('/content'); PROJECT_ROOT=ROOT/'Text-Mining---NewsQA-RAG'; WORK=ROOT/'phase3_abstention'; DATA=WORK/'data'; PREP=WORK/'phase2b_preparation'; OUTPUT=WORK/'results'
for path in [DATA,PREP,OUTPUT]: path.mkdir(parents=True,exist_ok=True)
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing Phase 3 files'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
HF_TOKEN=userdata.get('HF_TOKEN') or ''; GEMINI_API_KEY=userdata.get('GEMINI_API_KEY_1') or ''
os.environ.update({'GEMINI_API_KEY':GEMINI_API_KEY,'HF_HOME':str(ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1'})
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def run(args): print('$',' '.join(map(str,args)),flush=True); subprocess.run(list(map(str,args)),cwd=PROJECT_ROOT,check=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})

In [ ]:
from huggingface_hub import hf_hub_download
artifact=DATA/'locked-bge-m3-512-64-deduplicated-v2'
if not (artifact/'bundle_manifest.json').exists():
    bundle=Path(hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=HF_TOKEN or None)); assert sha(bundle)==HF_ARTIFACT_SHA256; shutil.unpack_archive(bundle,artifact)
if not (PREP/'results/preparation_bundle_manifest.json').exists():
    assert HF_TOKEN; bundle=Path(hf_hub_download(repo_id=HF_PREPARATION_REPO_ID,repo_type='dataset',revision=HF_PREPARATION_REVISION,filename=HF_PREPARATION_FILENAME,token=HF_TOKEN)); assert sha(bundle)==HF_PREPARATION_SHA256; shutil.unpack_archive(bundle,PREP)
source=Path(ABSTENTION_DATASET_PATH); assert source.exists(), 'Set ABSTENTION_DATASET_PATH to the finalized artifact'
cases_root=WORK/'cases_final'
if source.is_file(): shutil.unpack_archive(source,cases_root)
else: shutil.copytree(source,cases_root,dirs_exist_ok=True)
matches=list(cases_root.rglob('development_cases.jsonl')); assert len(matches)==1; cases_root=matches[0].parent
assert (cases_root/'final_test_cases.jsonl').exists() and (cases_root/'validation_report.json').exists()
validation=json.loads((cases_root/'validation_report.json').read_text()); assert validation['status']=='passed'
config_data=yaml.safe_load((PREP/'index/phase2b_config.yaml').read_text()); config_data['llm'].update({'model':'gemini-3.1-flash-lite','temperature':0.0,'max_tokens':512,'reasoning_effort':'minimal'})
config=WORK/'phase3_config.yaml'; config.write_text(yaml.safe_dump(config_data,sort_keys=False))
phase2_prompt=PREP/f'prompts/{PHASE2_WINNER_PROMPT_ID}.txt'; assert phase2_prompt.exists()
print('Cases:',cases_root,'| prompt:',PHASE2_WINNER_PROMPT_ID,'| depth:',CONTEXT_DEPTH)

In [ ]:
assert RUN_STAGE in {'smoke','development','final'}
assert EXECUTE, 'Review the locked inputs, then set EXECUTE=True'
assert GEMINI_API_KEY, 'Add GEMINI_API_KEY_1 to Colab secrets'
stage='development' if RUN_STAGE=='smoke' else RUN_STAGE
stage_output=OUTPUT/RUN_STAGE
command=[sys.executable,'scripts/run_phase3_abstention.py','--stage',stage,'--cases-root',cases_root,'--chunks',artifact/'chunks.jsonl','--sparse-index',artifact/'bge_m3_sparse.pkl','--config',config,'--phase2-prompt',phase2_prompt,'--output-dir',stage_output,'--context-depth',str(CONTEXT_DEPTH),'--generation-min-interval-seconds','4.2']
if RUN_STAGE=='smoke': command += ['--n-eval','7','--bootstrap-repetitions','100']
if RUN_STAGE=='final':
    assert WINNER_DECISION_PATH and Path(WINNER_DECISION_PATH).is_file(), 'Attach the locked development winner decision'
    command += ['--winner-decision',WINNER_DECISION_PATH]
    if THRESHOLD_DECISION_PATH: command += ['--threshold-decision',THRESHOLD_DECISION_PATH]
run(command)

In [ ]:
bundle=shutil.make_archive(str(ROOT/f'phase3_abstention_{RUN_STAGE}_results'),'zip',stage_output)
drive_target=Path('/content/drive/MyDrive/newsqa_phase3')/Path(bundle).name; drive_target.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(bundle,drive_target)
print('Saved:',drive_target,round(drive_target.stat().st_size/2**20,1),'MiB')